# Gas Compressor Investment, PLTG 4x25 MW Maleo-Gorontalo: Where Does a US$7.1M Capex Decision Break?
**Case Study #11 in *The Feasibility Stress Test*. Primary lens: Capex Planning.**

Reconstruction of Siregar, A. & Wiryono, S. (2021). *Implementation Monte Carlo Simulation in Investment Evaluation: Case Study - Gas Compressor Investment PLTG 4 x 25 MW Maleo-Gorontalo.* ICAESS 2021, pp. 72-79. DOI 10.5220/0010935100003255.

## Honesty disclosure (read first)
* The full text was read through automated extraction of the SCITEPRESS page, not a manual PDF review. Verify numbers against the paper before reuse.
* The paper does **not** publish its annual cash flows, tariff, tax rate, depreciation, the numeric Monte Carlo bounds, or the triangular mode.
* So this notebook is **not** a cash-flow rebuild. It is a **response-surface reconstruction**: NPV is modelled as the paper's base NPV plus three additive effects, each interpolated through the points of the paper's own one-at-a-time sensitivity table (Table 7). The Monte Carlo bounds are the **sensitivity-table ranges**, which is my assumption.
* Additivity (no interaction between drivers) is also an assumption. Treat simulated risk figures as method, not as evidence about the real project.

In [1]:
import numpy as np, pandas as pd, json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(11)

# --- CONFIRMED from the paper ---
CAPEX = 7_118_174          # USD incl. 10% PPN (6,471,067 + 647,107)
BASE = dict(npv=500_444.15, irr=0.1150, pi=1.08, payback=1.57)
PAPER_MC = dict(mean=474_346.30, median=481_496.58, sd=375_428.74, skew=-0.03, kurt=-0.56,
                min=-449_661.21, max=1_434_177.00, p_loss=0.1032, iters=1000)
# Table 7 one-at-a-time sensitivity (NPV, USD)
CAPEX_SENS = {-0.10: 968_869, 0.0: BASE['npv'], 0.10: 32_019.20}
LIFE_SENS  = {18: 93_955, 24: BASE['npv'], 30: 879_563}       # months
DE_SENS    = {0.60: 350_429, 0.70: BASE['npv'], 0.80: 655_908}  # debt share (60:40, 70:30, 80:20)
print('NPV / capex =', round(BASE['npv']/CAPEX*100,2), '%')

NPV / capex = 7.03 %


## 1. Response surface from the paper's sensitivity table
Piecewise-linear interpolation through the three published points per driver. Effects add.

In [2]:
def eff(table):
    xs = np.array(sorted(table)); ys = np.array([table[x]-BASE['npv'] for x in xs])
    return lambda x: np.interp(x, xs, ys)
f_capex = eff(CAPEX_SENS); f_life = eff(LIFE_SENS); f_de = eff(DE_SENS)
def npv(capex_mult=1.0, life=24, debt=0.70):
    return BASE['npv'] + f_capex(capex_mult-1) + f_life(life) + f_de(debt)

# Check: reproduces the paper's one-at-a-time points by construction
chk = pd.DataFrame([
  ['Capex -10%', npv(0.9), 968_869], ['Capex +10%', npv(1.1), 32_019.2],
  ['Life 18 mo', npv(life=18), 93_955], ['Life 30 mo', npv(life=30), 879_563],
  ['D:E 80:20', npv(debt=0.8), 655_908], ['D:E 60:40', npv(debt=0.6), 350_429]],
  columns=['Case','Reconstruction','Paper'])
print(chk.round(0).to_string(index=False))
# All-worst / all-best corners vs paper
worst = npv(1.1,18,0.6); best = npv(0.9,30,0.8)
print('\nAdditive all-worst corner:', round(worst), '| paper MC minimum:', PAPER_MC['min'], '| paper "Worst" scenario:', 199_076)
print('Additive all-best  corner:', round(best), '| paper MC maximum:', PAPER_MC['max'], '| paper "Best" scenario:', 1_239_437)

      Case  Reconstruction    Paper
Capex -10%        968869.0 968869.0
Capex +10%         32019.0  32019.0
Life 18 mo         93955.0  93955.0
Life 30 mo        879563.0 879563.0
 D:E 80:20        655908.0 655908.0
 D:E 60:40        350429.0 350429.0

Additive all-worst corner: -524485 | paper MC minimum: -449661.21 | paper "Worst" scenario: 199076
Additive all-best  corner: 1503452 | paper MC maximum: 1434177.0 | paper "Best" scenario: 1239437


## 2. Break-even points (own arithmetic, linear interpolation on the paper's table)

In [3]:
be_capex = BASE['npv']/ (-(f_capex(0.10))) * 0.10
slope_life = (BASE['npv']-LIFE_SENS[18])/6
be_life = 24 - BASE['npv']/slope_life
print(f'Capex overrun that takes NPV to zero: +{be_capex*100:.1f}%  (= US${be_capex*CAPEX:,.0f})')
print(f'Capex +10% = US${0.1*CAPEX:,.0f} -> removes {(BASE["npv"]-CAPEX_SENS[0.10])/BASE["npv"]*100:.1f}% of NPV')
print(f'Contract life that takes NPV to zero: {be_life:.1f} months (linear below 24 mo, extrapolated past the 18 mo data point)')
print(f'Base NPV is {BASE["npv"]/CAPEX*100:.1f}% of capex')
print('NPV swing per driver across the sensitivity ranges (USD):')
sw = {'Capex +/-10%': CAPEX_SENS[-0.1]-CAPEX_SENS[0.1], 'Life 18-30 mo': LIFE_SENS[30]-LIFE_SENS[18], 'D:E 60:40-80:20': DE_SENS[0.8]-DE_SENS[0.6]}
for k,v in sw.items(): print(f'  {k:18s} {v:>10,.0f}')

Capex overrun that takes NPV to zero: +10.7%  (= US$760,474)
Capex +10% = US$711,817 -> removes 93.6% of NPV
Contract life that takes NPV to zero: 16.6 months (linear below 24 mo, extrapolated past the 18 mo data point)
Base NPV is 7.0% of capex
NPV swing per driver across the sensitivity ranges (USD):
  Capex +/-10%          936,850
  Life 18-30 mo         785,608
  D:E 60:40-80:20       305,479


## 3. Monte Carlo
**Assumed bounds (not published):** capex multiplier U(0.90, 1.10); life U(18, 30) months; debt share Triangular(0.60, 0.70, 0.80). The paper states uniform for capex and life and triangular for D:E, 1,000 iterations. I use 20,000 trials for a stable estimate and also report a 1,000-trial run.

In [4]:
def simulate(n, capex_fixed=False, life_fixed=False, seed=11):
    r = np.random.default_rng(seed)
    cm = np.ones(n) if capex_fixed else r.uniform(0.9,1.1,n)
    lf = np.full(n,24.0) if life_fixed else r.uniform(18,30,n)
    de = r.triangular(0.6,0.7,0.8,n)
    return npv(cm,lf,de), cm, lf, de
def summ(v):
    return dict(mean=v.mean(), median=np.median(v), sd=v.std(ddof=1), skew=stats.skew(v), kurt=stats.kurtosis(v),
                min=v.min(), max=v.max(), p5=np.percentile(v,5), p95=np.percentile(v,95), p_loss=(v<0).mean())
v, cm, lf, de = simulate(20000)
s20k = summ(v); s1k = summ(simulate(1000, seed=3)[0])
tab = pd.DataFrame({'Paper (1,000 iter)':PAPER_MC, 'Recon 20,000':s20k, 'Recon 1,000':s1k}).drop(['iters'])
print(tab.round(3).to_string())

        Paper (1,000 iter)  Recon 20,000  Recon 1,000
mean            474346.300    497783.624   492930.657
median          481496.580    495209.023   503993.580
sd              375428.740    359731.209   371973.541
skew                -0.030        -0.004       -0.043
kurt                -0.560        -0.617       -0.728
min            -449661.210   -444080.172  -347054.686
max            1434177.000   1471301.346  1313166.700
p_loss               0.103         0.093        0.100
p5                     NaN   -101991.161  -137702.033
p95                    NaN   1100068.804  1089789.944


## 4. Which driver carries the risk? Variance share (Spearman) and what a board could fix

In [5]:
rho = {k: stats.spearmanr(x, v)[0] for k,x in {'Capex':cm,'Contract life':lf,'Debt share':de}.items()}
var_share = {k: r**2/sum(q**2 for q in rho.values()) for k,r in rho.items()}
print('Spearman rho:', {k:round(x,3) for k,x in rho.items()})
print('Share of explained variance (rho^2 normalised):', {k:round(x,3) for k,x in var_share.items()})

# Counterfactual levers (same seed): fix capex at base (fixed-price EPC), or guarantee life at 24 months
lev = {}
lev['As modelled'] = summ(simulate(20000)[0])
lev['Capex fixed at base'] = summ(simulate(20000, capex_fixed=True)[0])
lev['Life guaranteed at 24 mo'] = summ(simulate(20000, life_fixed=True)[0])
lev['Both fixed'] = summ(simulate(20000, capex_fixed=True, life_fixed=True)[0])
levdf = pd.DataFrame(lev).T[['mean','sd','p5','p_loss']]
print(levdf.round(3).to_string())

# Systematic capex bias: what if the real overrun centre is +5%, +10%?
bias_rows=[]
for b in [0,0.05,0.10]:
    r = np.random.default_rng(11)
    cmb = 1+b+r.uniform(-0.1,0.1,20000); lfb = r.uniform(18,30,20000); deb = r.triangular(0.6,0.7,0.8,20000)
    vb = npv(cmb,lfb,deb); bias_rows.append([f'+{int(b*100)}%', vb.mean(), (vb<0).mean()])
bias = pd.DataFrame(bias_rows, columns=['Capex bias','Mean NPV','P(NPV<0)'])
print(); print(bias.round(3).to_string(index=False))
print('(Capex multipliers above 1.10 extrapolate the paper\'s +/-10% sensitivity line.)')

Spearman rho: {'Capex': np.float64(-0.756), 'Contract life': np.float64(0.622), 'Debt share': np.float64(0.172)}
Share of explained variance (rho^2 normalised): {'Capex': np.float64(0.578), 'Contract life': np.float64(0.392), 'Debt share': np.float64(0.03)}
                                mean          sd          p5  p_loss
As modelled               497783.624  359731.209 -101991.161   0.093
Capex fixed at base       492144.561  235023.583  121871.059   0.002
Life guaranteed at 24 mo  504277.540  277606.148   68994.801   0.011
Both fixed                500746.459   62196.995  398151.245   0.000

Capex bias   Mean NPV  P(NPV<0)
       +0% 497783.624     0.093
       +5% 292372.502     0.209
      +10% 145215.817     0.325
(Capex multipliers above 1.10 extrapolate the paper's +/-10% sensitivity line.)


## 5. Chart

In [6]:
fig, ax = plt.subplots(1,2, figsize=(12,4.2))
ax[0].hist(v/1e6, bins=50, color='#805ad5', alpha=.85)
for x,c,l in [(0,'#c53030','break-even'),(BASE['npv']/1e6,'#d69e2e','paper base NPV'),(v.mean()/1e6,'#2f855a','simulated mean')]:
    ax[0].axvline(x,color=c,ls='--',label=l)
ax[0].set_xlabel('NPV (US$ million)'); ax[0].set_ylabel('trials'); ax[0].set_title('Reconstructed NPV distribution (illustrative)'); ax[0].legend()
labs=list(sw); ax[1].barh(labs,[sw[k]/1e6 for k in labs],color='#2b6cb0')
ax[1].set_xlabel('NPV swing across the paper\'s sensitivity range (US$M)'); ax[1].set_title('Capex and life dominate')
plt.tight_layout(); plt.savefig('chart.png',dpi=110); plt.show()

## 6. Save outputs

In [7]:
out = dict(
  citation="Siregar & Wiryono (2021), ICAESS 2021, pp.72-79, DOI 10.5220/0010935100003255",
  method="response-surface reconstruction from paper's Table 7; NOT a cash-flow model; MC bounds assumed",
  paper_mc=PAPER_MC, reconstruction_20000=s20k, reconstruction_1000=s1k,
  break_even=dict(capex_overrun_pct=be_capex*100, capex_overrun_usd=be_capex*CAPEX, life_months=be_life),
  npv_to_capex_pct=BASE['npv']/CAPEX*100, spearman=rho, variance_share=var_share,
  levers={k:{kk:float(vv) for kk,vv in d.items()} for k,d in lev.items()},
  capex_bias=bias.to_dict('records'),
  corners=dict(all_worst=float(worst), all_best=float(best)))
json.dump(out, open('notebook_reconstruction_output.json','w'), indent=2, default=float)
print('saved')

saved
